# Clinical Severity-Weighted Hallucination Score (CWHS)

Post-hoc analysis notebook that computes the **Clinical Severity-Weighted Hallucination Score (CWHS)**
across all RAG architectures evaluated in this study.

**Core idea:** Standard hallucination evaluation (e.g. DeepEval FaithfulnessMetric) treats all
questions equally. In medical QA, a hallucinated drug dosage is far more dangerous than a hallucinated
definition. CWHS applies clinical risk weights to the per-question hallucination rate derived from
FaithfulnessMetric, producing a severity-adjusted score that penalises architectures that hallucinate
more on the highest-risk question types.

**Formula:** `CWHS = Σ[(1 - f_i) × w_i] / Σ[w_i]`  
where `f_i` = FaithfulnessMetric score, `w_i` = severity weight (High=3, Medium=2, Low=1)

**Severity classification:** Two-stage hybrid — keyword matching (Stage 1) with LLM fallback
for questions that match no keywords (Stage 2). Labels are cached to
`datasets/processed/golden_dataset_with_severity.csv` and reused across all architectures.

In [5]:
import sys
sys.path.append("..")

import os
import time
import glob
from ast import literal_eval
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
import config

import logging
logging.basicConfig(level=logging.ERROR)

## Severity Classification Constants

Clinical risk taxonomy for biomedical questions:
- **High (weight=3)**: Treatment, medication, dosage, surgery — wrong answer could directly harm a patient
- **Medium (weight=2)**: Diagnosis, symptoms, prognosis, risk factors — wrong answer could mislead clinical reasoning
- **Low (weight=1)**: Definitions, mechanisms, epidemiology — primarily educational, indirect patient impact

In [6]:
HIGH_RISK_KEYWORDS = [
    "drug", "dose", "dosage", "medication", "prescri", "antibiotic",
    "treat", "treatment", "therapy", "therapies", "intervention", "regimen",
    "surgery", "surgical", "procedure", "adverse", "vaccine", "vaccination",
]

MEDIUM_RISK_KEYWORDS = [
    "diagnos", "symptom", "prognosis", "outcome", "risk factor", "risk of",
    "predict", "indicator", "biomarker", "complication",
    "disease", "disorder", "syndrome", "condition", "side effect",
]

SEVERITY_WEIGHTS = {"High": 3, "Medium": 2, "Low": 1}

## Stage 1: Keyword-Based Classification

In [7]:
def classify_by_keyword(question: str):
    """Stage 1: classify question severity using keyword substring matching.

    Returns:
        Tuple of (tier str, source str) where source is 'keyword' if matched,
        or 'pending_llm' if no keywords matched (requires Stage 2).
    """
    q = question.lower()
    if any(kw in q for kw in HIGH_RISK_KEYWORDS):
        return "High", "keyword"
    if any(kw in q for kw in MEDIUM_RISK_KEYWORDS):
        return "Medium", "keyword"
    return "Low", "pending_llm"

## Stage 2: LLM Fallback for Keyword-Unmatched Questions

Any question that matched no keywords in Stage 1 is classified by a single LLM call.
This handles biomedical phrasing that does not surface the expected keywords
(Latin terminology, abbreviations, paraphrases). The LLM is given the same
three-tier taxonomy with concrete clinical examples for each tier.

Stage 2 calls are parallelised across API keys using the same `ThreadPoolExecutor`
pattern used throughout the project.

In [8]:
SEVERITY_LLM_PROMPT_TEMPLATE = """You are a clinical risk assessor for medical AI systems.
Classify the following biomedical question into exactly one clinical risk tier.

Risk tiers:
- High: The question concerns drug dosage, medication choice, treatment protocol, surgical
  procedure, or any clinical decision that directly affects patient care. A wrong answer
  could cause direct patient harm.
  Examples: "Which antibiotic should be used?", "What is the standard dose of metformin?",
  "Is surgery indicated for this condition?"

- Medium: The question concerns diagnosis, symptom interpretation, prognosis, risk factors,
  or clinical outcome prediction. A wrong answer could mislead clinical reasoning but
  does not directly prescribe an action.
  Examples: "What are the early symptoms of Parkinson's?", "Does smoking increase the risk
  of this disease?", "What is the prognosis for stage 2 lung cancer?"

- Low: The question concerns definitions, biological mechanisms, epidemiology, or general
  scientific knowledge. A wrong answer is primarily misleading in an educational context.
  Examples: "What is the mechanism of action of aspirin?", "How common is type 1 diabetes?",
  "What is the role of the hippocampus in memory?"

Return only one word: High, Medium, or Low. Do not explain.

Question: {question}

Risk tier:"""

SEVERITY_LLM_PROMPT = PromptTemplate(
    template=SEVERITY_LLM_PROMPT_TEMPLATE,
    input_variables=["question"],
)


class GroqKeyRotator:
    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        print(f"Initialized GroqKeyRotator with {len(self.api_keys)} API key(s)")

    def get_llm(self):
        return ChatGroq(model=self.model, api_key=self.api_keys[0])


def _classify_slice_llm(questions_with_idx, api_key, model, delay, key_idx):
    """Classify a slice of questions via LLM. Returns list of (idx, tier) tuples."""
    llm = ChatGroq(model=model, api_key=api_key)
    chain = SEVERITY_LLM_PROMPT | llm
    results = []
    for i, (orig_idx, question) in enumerate(questions_with_idx):
        try:
            raw = chain.invoke({"question": question}).content.strip()
            tier = raw if raw in ("High", "Medium", "Low") else "Low"
        except Exception as e:
            print(f"[Key {key_idx}] Error on question {orig_idx}: {e}")
            tier = "Low"
        results.append((orig_idx, tier))
        if i < len(questions_with_idx) - 1:
            time.sleep(delay)
    print(f"[Key {key_idx}] Done — {len(results)} questions classified")
    return results


def classify_llm_parallel(questions_with_idx, key_rotator, rows_per_key=None, delay=None):
    """Classify questions via LLM in parallel across API keys.

    Args:
        questions_with_idx: List of (original_index, question_text) tuples.
        key_rotator: GroqKeyRotator instance.
        rows_per_key: Max questions per key (default config.PARALLEL_BUCKET_SIZE).
        delay: Seconds between questions per key.

    Returns:
        Dict mapping original_index → tier string.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or 1  # LLM classification is a short call; 1s delay is sufficient
    api_keys = key_rotator.api_keys

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(questions_with_idx):
            break
        slices.append((key, i, questions_with_idx[start: start + rows_per_key]))

    print(f"\n{len(questions_with_idx)} questions for LLM fallback, split across {len(slices)} key(s):")

    ordered_results = [None] * len(slices)
    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(_classify_slice_llm, s, key, key_rotator.model, delay, i): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            slice_idx = future_to_idx[future]
            try:
                ordered_results[slice_idx] = future.result()
            except Exception as e:
                print(f"[Key {slice_idx}] Thread failed: {e}")
                ordered_results[slice_idx] = []

    result_map = {}
    for key_results in ordered_results:
        for orig_idx, tier in key_results:
            result_map[orig_idx] = tier
    return result_map

## Hybrid Classifier + Caching

Runs Stage 1 (keyword) for all questions, then Stage 2 (LLM) for unmatched ones.
Results are saved to `datasets/processed/golden_dataset_with_severity.csv` and
reloaded on subsequent runs — the LLM calls only fire once for the 200-question set.

In [9]:
SEVERITY_CACHE_PATH = config.DATA_PROCESSED_DIR / "golden_dataset_with_severity.csv"


def build_severity_labels(golden_df, key_rotator, force_recompute=False):
    """Classify all questions by clinical severity using the hybrid approach.

    Loads from cache if it exists (unless force_recompute=True).
    Stage 1: keyword matching for all questions.
    Stage 2: LLM fallback for questions that matched no keywords.

    Args:
        golden_df: DataFrame with at least a 'question' column.
        key_rotator: GroqKeyRotator for Stage 2 LLM calls.
        force_recompute: If True, ignore cache and re-run classification.

    Returns:
        DataFrame with added columns: severity_tier, severity_weight, classification_source.
    """
    if not force_recompute and SEVERITY_CACHE_PATH.exists():
        print(f"Loading severity labels from cache: {SEVERITY_CACHE_PATH}")
        cached = pd.read_csv(SEVERITY_CACHE_PATH)
        return cached

    df = golden_df.copy().reset_index(drop=True)

    # Stage 1: keyword classification
    tiers, sources = [], []
    for q in df["question"]:
        tier, source = classify_by_keyword(q)
        tiers.append(tier)
        sources.append(source)

    df["severity_tier"] = tiers
    df["classification_source"] = sources

    pending_llm = df[df["classification_source"] == "pending_llm"]
    keyword_count = len(df) - len(pending_llm)
    print(f"Stage 1 (keyword): {keyword_count}/{len(df)} questions classified")
    print(f"Stage 2 (LLM fallback): {len(pending_llm)} questions to reclassify")

    # Stage 2: LLM reclassification for pending_llm questions
    if len(pending_llm) > 0:
        questions_with_idx = list(zip(pending_llm.index.tolist(), pending_llm["question"].tolist()))
        llm_results = classify_llm_parallel(questions_with_idx, key_rotator)
        for orig_idx, tier in llm_results.items():
            df.at[orig_idx, "severity_tier"] = tier
            df.at[orig_idx, "classification_source"] = "llm_fallback"

    df["severity_weight"] = df["severity_tier"].map(SEVERITY_WEIGHTS)

    # Cache results
    df.to_csv(SEVERITY_CACHE_PATH, index=False)
    print(f"\nSeverity labels saved to {SEVERITY_CACHE_PATH}")

    # Summary
    print(f"\nSeverity distribution:")
    for tier in ["High", "Medium", "Low"]:
        n = (df["severity_tier"] == tier).sum()
        print(f"  {tier} (w={SEVERITY_WEIGHTS[tier]}): {n} questions ({100*n/len(df):.1f}%)")
    print(f"\nClassification source:")
    print(df["classification_source"].value_counts().to_string())

    return df

## CWHS Computation

In [10]:
def compute_cwhs(severity_df, faithfulness_scores):
    """Compute Clinical Severity-Weighted Hallucination Score.

    CWHS = Σ[(1 - f_i) × w_i] / Σ[w_i]

    Args:
        severity_df: DataFrame with severity_tier and severity_weight columns,
            aligned by index to faithfulness_scores.
        faithfulness_scores: List or Series of FaithfulnessMetric scores (0–1,
            higher = less hallucination).

    Returns:
        Dict with:
          cwhs: float [0-1], severity-weighted hallucination score
          unweighted_hal_rate: float [0-1], plain 1 - mean(faithfulness)
          delta: cwhs - unweighted_hal_rate (positive = worse on high-risk Qs)
          mean_faithfulness: float, mean faithfulness score
          n_questions: int, number of questions evaluated
          severity_breakdown: per-tier dict with count and mean hallucination rate
    """
    weights = list(severity_df["severity_weight"])
    tiers = list(severity_df["severity_tier"])
    scores = list(faithfulness_scores)

    assert len(weights) == len(scores), "Mismatch between severity labels and faithfulness scores"

    hal_rates = [1 - f for f in scores]
    total_weight = sum(weights)
    cwhs = sum(h * w for h, w in zip(hal_rates, weights)) / total_weight
    unweighted = sum(hal_rates) / len(hal_rates)

    breakdown = {}
    for tier in ["High", "Medium", "Low"]:
        indices = [i for i, t in enumerate(tiers) if t == tier]
        if indices:
            tier_hal = [hal_rates[i] for i in indices]
            breakdown[tier] = {
                "count": len(indices),
                "mean_hal_rate": round(sum(tier_hal) / len(tier_hal), 4),
                "weight": SEVERITY_WEIGHTS[tier],
            }
        else:
            breakdown[tier] = {"count": 0, "mean_hal_rate": None, "weight": SEVERITY_WEIGHTS[tier]}

    return {
        "cwhs": round(cwhs, 4),
        "unweighted_hal_rate": round(unweighted, 4),
        "delta": round(cwhs - unweighted, 4),
        "mean_faithfulness": round(sum(scores) / len(scores), 4),
        "n_questions": len(scores),
        "severity_breakdown": breakdown,
    }

## Architecture Registry

Maps a human-readable architecture name to its DeepEval Faithfulness CSV file path.
Update this dict with actual timestamped filenames after each experiment run.
Any architecture whose file path does not exist is skipped gracefully.

In [20]:
DEEPEVAL_DIR = config.RESULTS_DEEPEVAL_DIR

# Map architecture label → faithfulness CSV filename (relative to DEEPEVAL_DIR)
# Fill in the actual timestamps after running each experiment
ARCHITECTURE_FAITHFULNESS_FILES = {
    "Naive RAG k=3":                 "naive_rag_minilm_faithfulness_20260504_132359.csv",
    "Naive RAG k=5":                 "naive_rag_minilm_k_5_faithfulness_20260506_145734.csv",
    "Naive RAG k=8":                 "naive_rag_minilm_k_8_faithfulness_20260507_145335.csv",
    "Query Expansion (Single)":       "query_expansion_rag_minilm_faithfulness_20260509_121158.csv",
    "Query Expansion (Multi)":        "multi_query_expansion_rag_minilm_faithfulness_20260509_155059.csv",
    "Hybrid RRF":                     "hybrid_rrf_rag_minilm_faithfulness_20260510_081037.csv",
    "Hybrid + Cross-Encoder":         "hybrid_cross_encoder_rag_minilm_faithfulness_20260510_062636.csv",
    "MeSH-Guided RAG":                "mesh_guided_rag_minilm_faithfulness_20260512_132445.csv",
    "Evidence-Graded RAG":            "evidence_graded_rag_minilm_faithfulness_20260511_140805.csv",
}


def load_faithfulness_csv(filename):
    """Load a DeepEval faithfulness CSV and return (questions list, scores list).

    Handles two possible column names: 'Faithfulness' and 'FaithfulnessMetric'.
    Returns None if the file does not exist.
    """
    path = DEEPEVAL_DIR / filename
    if not path.exists():
        return None
    df = pd.read_csv(path)
    # Normalise column name
    score_col = next((c for c in df.columns if "faithfulness" in c.lower()), None)
    if score_col is None:
        print(f"Warning: no faithfulness score column found in {filename}")
        return None
    return df["question"].tolist(), df[score_col].tolist()


print("Architecture registry configured.")
print(f"\nLooking for faithfulness results in: {DEEPEVAL_DIR}")

Architecture registry configured.

Looking for faithfulness results in: /content/results/deepeval


In [22]:
# open all files from ARCHITECTURE_FAITHFULNESS_FILES and check if len is 200
for arch_name, filename in ARCHITECTURE_FAITHFULNESS_FILES.items():
    loaded = load_faithfulness_csv(filename)
    if loaded is None:
        continue
    questions, scores = loaded
    print(f"{arch_name}: {len(questions)} questions, {len(scores)} scores")
    assert len(questions) == len(scores) == 200


Naive RAG k=3: 200 questions, 200 scores
Naive RAG k=5: 200 questions, 200 scores
Naive RAG k=8: 200 questions, 200 scores
Query Expansion (Single): 200 questions, 200 scores
Query Expansion (Multi): 200 questions, 200 scores
Hybrid RRF: 200 questions, 200 scores
Hybrid + Cross-Encoder: 200 questions, 200 scores
MeSH-Guided RAG: 200 questions, 200 scores
Evidence-Graded RAG: 200 questions, 200 scores


## Setup: Load Golden Dataset and Run Severity Classification

In [16]:
golden_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_complete.csv")
golden_df['golden_contexts'] = golden_df['golden_contexts'].apply(literal_eval)
print(f"Loaded {len(golden_df)} questions from golden_dataset_complete.csv")
golden_df.head(3)

Loaded 200 questions from golden_dataset_complete.csv


,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed
0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841']
1,1,"Do the changes in the serum levels of IL-2, IL...",The enhancement of serum TNFalpha and IL-6 lev...,[Acute pancreatitis is the major complication ...,Single-hop,['18670651']
2,2,Does hypoglycaemia increase the risk of cardio...,Severe hypoglycaemia is associated with an inc...,[Hypoglycaemia caused by glucose-lowering ther...,Single-hop,['23999452']


In [ ]:
# Run severity classification (cached after first run)
key_rotator = GroqKeyRotator()
severity_df = build_severity_labels(golden_df, key_rotator)

Initialized GroqKeyRotator with 10 API key(s)
Stage 1 (keyword): 104/200 questions classified
Stage 2 (LLM fallback): 96 questions to reclassify

96 questions for LLM fallback, split across 5 key(s):
[Key 4] Done — 16 questions classified
[Key 3] Done — 20 questions classified
[Key 1] Done — 20 questions classified
[Key 0] Done — 20 questions classified
[Key 2] Done — 20 questions classified

Severity labels saved to /content/drive/MyDrive/LJMU/processed/golden_dataset_with_severity.csv

Severity distribution:
  High (w=3): 56 questions (28.0%)
  Medium (w=2): 122 questions (61.0%)
  Low (w=1): 22 questions (11.0%)

Classification source:
classification_source
keyword         104
llm_fallback     96


In [ ]:
# Inspect the severity distribution
print("Severity distribution:")
print(severity_df["severity_tier"].value_counts().to_string())
print("\nClassification source:")
print(severity_df["classification_source"].value_counts().to_string())
severity_df[["question", "severity_tier", "severity_weight", "classification_source"]].head(10)

Severity distribution:
severity_tier
Medium    122
High       56
Low        22

Classification source:
classification_source
keyword         104
llm_fallback     96


,question,severity_tier,severity_weight,classification_source
0,Is there a relationship between rheumatoid art...,Medium,2,keyword
1,"Do the changes in the serum levels of IL-2, IL...",Medium,2,llm_fallback
2,Does hypoglycaemia increase the risk of cardio...,Medium,2,keyword
3,Telemedicine and type 1 diabetes: is technolog...,Medium,2,llm_fallback
4,Can elevated troponin I levels predict complic...,Medium,2,keyword
5,Does β-catenin have a role in pathogenesis of ...,Low,1,llm_fallback
6,Remote ischemic postconditioning: does it prot...,Medium,2,keyword
7,Hearing loss: an unknown complication of pre-e...,Medium,2,keyword
8,Does psychological distress predict disability?,Medium,2,keyword
9,Pancreas retransplantation: a second chance f...,High,3,llm_fallback


## Compute CWHS for All Architectures

For each architecture in the registry:
1. Load its faithfulness CSV
2. Match questions to severity labels by question text
3. Compute CWHS using the aligned severity weights and faithfulness scores

In [18]:
severity_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_with_severity.csv")
severity_df['golden_contexts'] = severity_df['golden_contexts'].apply(literal_eval)
severity_df.head(3)

,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,severity_tier,classification_source,severity_weight
0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841'],Medium,keyword,2
1,1,"Do the changes in the serum levels of IL-2, IL...",The enhancement of serum TNFalpha and IL-6 lev...,[Acute pancreatitis is the major complication ...,Single-hop,['18670651'],Medium,llm_fallback,2
2,2,Does hypoglycaemia increase the risk of cardio...,Severe hypoglycaemia is associated with an inc...,[Hypoglycaemia caused by glucose-lowering ther...,Single-hop,['23999452'],Medium,keyword,2


In [23]:
def align_severity_to_faithfulness(severity_df, questions, faithfulness_scores):
    """Align severity labels to the faithfulness CSV row order.

    Matches rows by question text (case-insensitive strip). Returns a
    filtered severity DataFrame aligned to the faithfulness score list.
    Rows in the faithfulness CSV whose question is not found in severity_df
    are assigned a default Low severity with a warning.
    """
    sev_map = {
        q.strip().lower(): (tier, w)
        for q, tier, w in zip(
            severity_df["question"],
            severity_df["severity_tier"],
            severity_df["severity_weight"],
        )
    }
    aligned_tiers, aligned_weights, aligned_scores = [], [], []
    missing = 0
    for q, f in zip(questions, faithfulness_scores):
        key = q.strip().lower()
        if key in sev_map:
            tier, w = sev_map[key]
        else:
            tier, w = "Low", 1
            missing += 1
        aligned_tiers.append(tier)
        aligned_weights.append(w)
        aligned_scores.append(f)
    if missing:
        print(f"Warning: {missing} questions not found in severity_df — assigned Low by default")
    aligned_df = pd.DataFrame({"severity_tier": aligned_tiers, "severity_weight": aligned_weights})
    return aligned_df, aligned_scores


results = {}
skipped = []

for arch_name, filename in ARCHITECTURE_FAITHFULNESS_FILES.items():
    loaded = load_faithfulness_csv(filename)
    if loaded is None:
        skipped.append(arch_name)
        continue
    questions, scores = loaded
    aligned_sev_df, aligned_scores = align_severity_to_faithfulness(severity_df, questions, scores)
    cwhs_result = compute_cwhs(aligned_sev_df, aligned_scores)
    results[arch_name] = cwhs_result
    print(f"{arch_name}: CWHS={cwhs_result['cwhs']:.4f}, "
          f"Unweighted={cwhs_result['unweighted_hal_rate']:.4f}, "
          f"Δ={cwhs_result['delta']:+.4f}")

if skipped:
    print(f"\nSkipped (file not found): {', '.join(skipped)}")

Naive RAG k=3: CWHS=0.0148, Unweighted=0.0141, Δ=+0.0007
Naive RAG k=5: CWHS=0.0200, Unweighted=0.0204, Δ=-0.0003
Naive RAG k=8: CWHS=0.0137, Unweighted=0.0148, Δ=-0.0011
Query Expansion (Single): CWHS=0.0141, Unweighted=0.0145, Δ=-0.0003
Query Expansion (Multi): CWHS=0.0132, Unweighted=0.0119, Δ=+0.0013
Hybrid RRF: CWHS=0.0198, Unweighted=0.0186, Δ=+0.0013
Hybrid + Cross-Encoder: CWHS=0.0195, Unweighted=0.0180, Δ=+0.0014
MeSH-Guided RAG: CWHS=0.0211, Unweighted=0.0189, Δ=+0.0021
Evidence-Graded RAG: CWHS=0.0343, Unweighted=0.0322, Δ=+0.0021


## Summary Table: CWHS vs Unweighted Hallucination Rate

In [25]:
if results:
    summary_rows = []
    for arch, r in results.items():
        summary_rows.append({
            "Architecture": arch,
            "Mean Faithfulness": r["mean_faithfulness"],
            "Unweighted Hal. Rate": r["unweighted_hal_rate"],
            "CWHS": r["cwhs"],
            "Δ (CWHS − Unweighted)": r["delta"],
            "N Questions": r["n_questions"],
        })
    summary_df = pd.DataFrame(summary_rows).sort_values("CWHS")
    print("\nCWHS Summary (sorted by CWHS, lower is better):")
    print(summary_df.to_string(index=False))
    summary_df.to_csv(
        config.RESULTS_FIGURES_DIR / "cwhs_summary.csv", index=False
    )
    print(f"\nSaved summary to {config.RESULTS_FIGURES_DIR / 'cwhs_summary.csv'}")


CWHS Summary (sorted by CWHS, lower is better):
            Architecture  Mean Faithfulness  Unweighted Hal. Rate   CWHS  Δ (CWHS − Unweighted)  N Questions
 Query Expansion (Multi)             0.9881                0.0119 0.0132                 0.0013          200
           Naive RAG k=8             0.9852                0.0148 0.0137                -0.0011          200
Query Expansion (Single)             0.9855                0.0145 0.0141                -0.0003          200
           Naive RAG k=3             0.9859                0.0141 0.0148                 0.0007          200
  Hybrid + Cross-Encoder             0.9820                0.0180 0.0195                 0.0014          200
              Hybrid RRF             0.9814                0.0186 0.0198                 0.0013          200
           Naive RAG k=5             0.9796                0.0204 0.0200                -0.0003          200
         MeSH-Guided RAG             0.9811                0.0189 0.0211       

## Severity-Stratified Breakdown

Shows the hallucination rate within each severity tier per architecture.
The key finding here is whether architectures differ not just in overall hallucination
rate but in *where* they hallucinate — an architecture that hallucinates heavily on
High-risk questions is more clinically dangerous than one that makes errors on Low-risk questions.

In [26]:
if results:
    breakdown_rows = []
    for arch, r in results.items():
        row = {"Architecture": arch}
        for tier in ["High", "Medium", "Low"]:
            bd = r["severity_breakdown"].get(tier, {})
            row[f"{tier} Hal. Rate"] = bd.get("mean_hal_rate", None)
            row[f"{tier} N"] = bd.get("count", 0)
        row["CWHS"] = r["cwhs"]
        breakdown_rows.append(row)
    breakdown_df = pd.DataFrame(breakdown_rows).sort_values("CWHS")
    print("\nSeverity-Stratified Hallucination Rates:")
    print(breakdown_df.to_string(index=False))
    breakdown_df.to_csv(
        config.RESULTS_FIGURES_DIR / "cwhs_breakdown_by_severity.csv", index=False
    )
    print(f"\nSaved breakdown to {config.RESULTS_FIGURES_DIR / 'cwhs_breakdown_by_severity.csv'}")


Severity-Stratified Hallucination Rates:
            Architecture  High Hal. Rate  High N  Medium Hal. Rate  Medium N  Low Hal. Rate  Low N   CWHS
 Query Expansion (Multi)          0.0176      56            0.0114       122         0.0000     22 0.0132
           Naive RAG k=8          0.0095      56            0.0157       122         0.0227     22 0.0137
Query Expansion (Single)          0.0134      56            0.0143       122         0.0182     22 0.0141
           Naive RAG k=3          0.0197      56            0.0116       122         0.0136     22 0.0148
  Hybrid + Cross-Encoder          0.0221      56            0.0194       122         0.0000     22 0.0195
              Hybrid RRF          0.0210      56            0.0208       122         0.0000     22 0.0198
           Naive RAG k=5          0.0186      56            0.0208       122         0.0227     22 0.0200
         MeSH-Guided RAG          0.0361      56            0.0108       122         0.0205     22 0.0211
    

## Δ Interpretation

The Δ column (CWHS − Unweighted Hallucination Rate) is the key research signal:

- **Δ > 0**: The architecture hallucinates disproportionately *more* on high-risk questions than on low-risk ones. Clinical severity amplifies the hallucination impact. Clinically concerning.
- **Δ ≈ 0**: Hallucination is evenly distributed across severity tiers. No differential risk.
- **Δ < 0**: The architecture hallucinates less on high-risk questions than on low-risk ones. The clinical safety profile is better than the raw hallucination rate suggests.

In [30]:
if results:
    print("Delta interpretation per architecture:")
    print("-" * 60)
    for arch, r in sorted(results.items(), key=lambda x: x[1]["delta"]):
        delta = r["delta"]
        if delta > 0.002:
            label = "⚠ Worse on high-risk questions"
        elif delta < -0.002:
            label = "✓ Better on high-risk questions"
        else:
            label = "~ Evenly distributed"
        print(f"  {arch:35s}  Δ={delta:+.4f}  {label}")

Delta interpretation per architecture:
------------------------------------------------------------
  Naive RAG k=8                        Δ=-0.0011  ~ Evenly distributed
  Naive RAG k=5                        Δ=-0.0003  ~ Evenly distributed
  Query Expansion (Single)             Δ=-0.0003  ~ Evenly distributed
  Naive RAG k=3                        Δ=+0.0007  ~ Evenly distributed
  Query Expansion (Multi)              Δ=+0.0013  ~ Evenly distributed
  Hybrid RRF                           Δ=+0.0013  ~ Evenly distributed
  Hybrid + Cross-Encoder               Δ=+0.0014  ~ Evenly distributed
  MeSH-Guided RAG                      Δ=+0.0021  ⚠ Worse on high-risk questions
  Evidence-Graded RAG                  Δ=+0.0021  ⚠ Worse on high-risk questions
